# COSA+ Analysis

Parses COSA+ experiment logs, builds ablation figures, plots learned gates, and computes COSA+ improvement over COSA-F.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULT_DIRS = [PROJECT_ROOT / 'results' / 'ablation', PROJECT_ROOT / 'results' / 'main_exp']
LOG_PATTERN = re.compile(r'(?P<variant>original|vec_gate|rich_ctx|ctx_std_only|cosa_plus)_(?P<dataset>.+?)_(?P<model>[^_]+)_(?P<pred_len>\d+)\.txt$')
MSE_PATTERN = re.compile(r'MSE:\s*([0-9.]+)')
MAE_PATTERN = re.compile(r'MAE:\s*([0-9.]+)')

records = []
for result_dir in RESULT_DIRS:
    if not result_dir.exists():
        continue
    for path in sorted(result_dir.glob('*.txt')):
        match = LOG_PATTERN.match(path.name)
        if not match:
            continue
        text = path.read_text(errors='ignore')
        mse_match = MSE_PATTERN.search(text)
        mae_match = MAE_PATTERN.search(text)
        if not mse_match:
            continue
        record = match.groupdict()
        record['pred_len'] = int(record['pred_len'])
        record['mse'] = float(mse_match.group(1))
        record['mae'] = float(mae_match.group(1)) if mae_match else np.nan
        record['source'] = result_dir.name
        records.append(record)

df = pd.DataFrame(records)
df.sort_values(['source', 'dataset', 'model', 'pred_len', 'variant']).reset_index(drop=True)

In [ ]:
ablation = df[(df['source'] == 'ablation') & (df['dataset'] == 'ETTh1') & (df['model'] == 'DLinear')].copy()
variant_order = ['original', 'vec_gate', 'rich_ctx', 'ctx_std_only', 'cosa_plus']
horizons = sorted(ablation['pred_len'].unique())

fig, axes = plt.subplots(1, max(1, len(horizons)), figsize=(4 * max(1, len(horizons)), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, horizon in zip(axes, horizons):
    subset = ablation[ablation['pred_len'] == horizon].set_index('variant').reindex(variant_order)
    ax.bar(subset.index, subset['mse'], color=['#4C78A8', '#F58518', '#54A24B', '#B279A2', '#E45756'])
    ax.set_title(f'Horizon {horizon}')
    ax.set_xlabel('Variant')
    ax.tick_params(axis='x', rotation=35)
axes[0].set_ylabel('MSE')
fig.suptitle('Figure 1: COSA+ Ablation on ETTh1 / DLinear')
fig.tight_layout()
plt.show()

In [ ]:
gate_candidates = sorted((PROJECT_ROOT / 'results').glob('**/gate_cosa_plus_ETTh1_DLinear_720.npy'))
if gate_candidates:
    gate = np.load(gate_candidates[-1]).reshape(-1)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.bar(np.arange(1, len(gate) + 1), gate, width=1.0, color='#4C78A8')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('Figure 2: Learned COSA+ Gate tanh(g), ETTh1 / DLinear / L=720')
    ax.set_xlabel('Forecast step')
    ax.set_ylabel('tanh(g)')
    plt.show()
else:
    print('No COSA+ gate file found for ETTh1 / DLinear / 720 yet.')

In [ ]:
comparison = df[df['variant'].isin(['original', 'cosa_plus'])].copy()
pivot = comparison.pivot_table(index=['source', 'dataset', 'model', 'pred_len'], columns='variant', values='mse')
pivot['cosa_plus_improvement_pct'] = (pivot['original'] - pivot['cosa_plus']) / pivot['original'] * 100
pivot.reset_index().sort_values(['source', 'dataset', 'model', 'pred_len'])